In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries and load the original training dataset

import os
import json
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

LABEL_COLUMNS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

train_df = pd.read_csv(os.path.join(RAW_DIR, "train.csv"))

print("Training dataset shape:", train_df.shape)
print("\nColumns:", train_df.columns.tolist())
print("\nMissing values:", train_df.isnull().sum().sum())

In [ ]:
# Check the values present in each target label

for label in LABEL_COLUMNS:
    print(f"{label}: {sorted(train_df[label].unique())}")

In [ ]:
# Create normalised comment text for grouping

import re

def normalize_comment(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_df["group_text"] = train_df["comment_text"].apply(normalize_comment)

print("Original rows:", len(train_df))
print("Unique normalised comments:", train_df["group_text"].nunique())
print("Duplicate rows after normalisation:", train_df["group_text"].duplicated().sum())

In [ ]:
# Assign a unique group ID to each normalised comment

train_df["group_id"] = pd.factorize(train_df["group_text"])[0]

print("Unique groups:", train_df["group_id"].nunique())
print("Total rows:", len(train_df))
print("Largest group size:", train_df.groupby("group_id").size().max())

In [ ]:
# Create group-level labels for stratified splitting

group_labels = train_df.groupby("group_id")[LABEL_COLUMNS].max()

print("Group-level dataset shape:", group_labels.shape)
print("\nPositive groups per label:")
print(group_labels.sum())

In [ ]:
# Count groups by the number of positive labels

group_labels["positive_label_count"] = group_labels[LABEL_COLUMNS].sum(axis=1)

print("Group distribution by positive-label count:")
print(group_labels["positive_label_count"].value_counts().sort_index())

In [ ]:
# Summarise group-level multi-label counts

group_multi_label = (group_labels["positive_label_count"] > 1).sum()
group_toxic = (group_labels["positive_label_count"] > 0).sum()

print("Groups with at least one positive label:", group_toxic)
print("Groups with multiple positive labels:", group_multi_label)
print("Groups with no positive labels:", (group_labels["positive_label_count"] == 0).sum())

In [ ]:
# Prepare group IDs and multi-label targets for splitting

group_ids = group_labels.index.to_numpy()
y_groups = group_labels[LABEL_COLUMNS].to_numpy()

print("Group IDs shape:", group_ids.shape)
print("Group label matrix shape:", y_groups.shape)
print("Number of labels:", y_groups.shape[1])

In [ ]:
# Install iterative multi-label stratification

!pip install iterative-stratification

In [ ]:
# Split groups using iterative multi-label stratification

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# First split: 70% training and 30% temporary
split_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(split_1.split(group_ids, y_groups))

# Second split: divide temporary groups equally
temp_ids = group_ids[temp_idx]
temp_labels = y_groups[temp_idx]

split_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(split_2.split(temp_ids, temp_labels))

# Store group IDs for each partition
train_group_ids = group_ids[train_idx]
val_group_ids = temp_ids[val_idx]
test_group_ids = temp_ids[test_idx]

print("Training groups:", len(train_group_ids))
print("Validation groups:", len(val_group_ids))
print("Internal test groups:", len(test_group_ids))
print("Total groups:", len(train_group_ids) + len(val_group_ids) + len(test_group_ids))

In [ ]:
# Verify that groups do not overlap between partitions

train_groups = set(train_group_ids)
val_groups = set(val_group_ids)
test_groups = set(test_group_ids)

print("Train–Validation overlap:", len(train_groups & val_groups))
print("Train–Test overlap:", len(train_groups & test_groups))
print("Validation–Test overlap:", len(val_groups & test_groups))

In [ ]:
# Assign original records to the three partitions

original_columns = ["id", "comment_text"] + LABEL_COLUMNS

train_full = train_df[train_df["group_id"].isin(train_group_ids)][original_columns].copy()
val_df = train_df[train_df["group_id"].isin(val_group_ids)][original_columns].copy()
test_internal = train_df[train_df["group_id"].isin(test_group_ids)][original_columns].copy()

print("Training rows:", train_full.shape)
print("Validation rows:", val_df.shape)
print("Internal test rows:", test_internal.shape)
print("Total assigned rows:", len(train_full) + len(val_df) + len(test_internal))
print("Original rows:", len(train_df))

In [ ]:
# Compare positive label counts and rates across partitions

for name, df in [
    ("Training", train_full),
    ("Validation", val_df),
    ("Internal Test", test_internal)
]:
    print(f"\n{name}")
    print(pd.DataFrame({
        "Positive Count": df[LABEL_COLUMNS].sum(),
        "Positive Rate (%)": (df[LABEL_COLUMNS].mean() * 100).round(3)
    }))

In [ ]:
# Count positive and clean records in the training partition

train_full["any_toxic"] = train_full[LABEL_COLUMNS].max(axis=1)

positive_train = train_full[train_full["any_toxic"] == 1].copy()
clean_train = train_full[train_full["any_toxic"] == 0].copy()

print("Total training records:", len(train_full))
print("Positive records:", len(positive_train))
print("Clean records:", len(clean_train))
print("Positive-to-clean ratio before sampling:",
      round(len(positive_train) / len(clean_train), 4))

In [ ]:
# Undersample clean training records at a 1:2 ratio

clean_sample = clean_train.sample(
    n=len(positive_train) * 2,
    random_state=42
)

train_sub = pd.concat(
    [positive_train, clean_sample],
    axis=0
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sampled training records:", len(train_sub))
print("Positive records:", train_sub["any_toxic"].sum())
print("Clean records:", (train_sub["any_toxic"] == 0).sum())

In [ ]:
# Check label distribution after training-only sampling

sampling_summary = pd.DataFrame({
    "Positive Count": train_sub[LABEL_COLUMNS].sum(),
    "Positive Rate (%)": (train_sub[LABEL_COLUMNS].mean() * 100).round(3)
})

print(sampling_summary)

In [ ]:
# Save processed datasets

os.makedirs(PROCESSED_DIR, exist_ok=True)

train_full.drop(columns=["any_toxic"], errors="ignore").to_csv(
    os.path.join(PROCESSED_DIR, "train_full.csv"), index=False
)

val_df.to_csv(
    os.path.join(PROCESSED_DIR, "val.csv"), index=False
)

test_internal.to_csv(
    os.path.join(PROCESSED_DIR, "test_internal.csv"), index=False
)

train_sub.drop(columns=["any_toxic"], errors="ignore").to_csv(
    os.path.join(PROCESSED_DIR, "train_sub.csv"), index=False
)

print("All four processed datasets saved successfully.")

In [ ]:
# Verify saved processed datasets

for filename in ["train_full.csv", "val.csv", "test_internal.csv", "train_sub.csv"]:
    filepath = os.path.join(PROCESSED_DIR, filename)
    df = pd.read_csv(filepath)
    print(f"{filename}: {df.shape}")

In [ ]:
# Save preprocessing experiment summary

preprocessing_summary = {
    "random_seed": 42,
    "original_records": len(train_df),
    "unique_normalized_groups": int(group_labels.shape[0]),
    "train_full_records": len(train_full),
    "validation_records": len(val_df),
    "internal_test_records": len(test_internal),
    "positive_training_records": len(positive_train),
    "clean_training_records_before_sampling": len(clean_train),
    "clean_training_records_after_sampling": len(clean_sample),
    "sampled_training_records": len(train_sub),
    "positive_to_clean_ratio": "1:2",
    "sampling_method": "Training-only clean record undersampling",
    "group_overlap": 0
}

summary_path = os.path.join(
    PROJECT_DIR, "results", "experiment_logs",
    "preprocessing_summary.json"
)

with open(summary_path, "w") as file:
    json.dump(preprocessing_summary, file, indent=4)

print("Preprocessing summary saved:", summary_path)

In [ ]:
# Save label prevalence across all dataset partitions

prevalence_summary = pd.DataFrame({
    "Label": LABEL_COLUMNS,
    "Original (%)": train_df[LABEL_COLUMNS].mean().values * 100,
    "Full Training (%)": train_full[LABEL_COLUMNS].mean().values * 100,
    "Sampled Training (%)": train_sub[LABEL_COLUMNS].mean().values * 100,
    "Validation (%)": val_df[LABEL_COLUMNS].mean().values * 100,
    "Internal Test (%)": test_internal[LABEL_COLUMNS].mean().values * 100
})

prevalence_summary = prevalence_summary.round(3)

prevalence_path = os.path.join(
    PROJECT_DIR, "results", "tables",
    "T04_prevalence_comparison.csv"
)

prevalence_summary.to_csv(prevalence_path, index=False)

print(prevalence_summary.to_string(index=False))
print("\nSaved:", prevalence_path)

In [ ]:
# Verify training sampling integrity

original_positive_ids = set(positive_train["id"])
sampled_ids = set(train_sub["id"])
full_train_ids = set(train_full["id"])

print("All positive records retained:",
      original_positive_ids.issubset(sampled_ids))

print("Sampled records belong to training partition:",
      sampled_ids.issubset(full_train_ids))

print("Duplicate IDs in sampled training data:",
      train_sub["id"].duplicated().sum())